In [1]:
import numpy as np
import pandas as pd

1

In [2]:
def PC(df):
    """
    Calculate the pairwise correlation coefficients for each column in a DataFrame.

    """
    return df.corr()

2

In [3]:


def calculate_condition_index(df):
    # Step 1: Compute the correlation matrix
    corr_matrix = df.corr()

    # Step 2: Compute eigenvalues of the correlation matrix
    eigenvalues, _ = np.linalg.eig(corr_matrix)

    # Step 3: Sort eigenvalues in descending order (for clarity)
    sorted_eigenvalues = np.sort(eigenvalues)[::-1]

    # Step 4: Compute condition index as sqrt(largest_eigenvalue / current_eigenvalue)
    max_eigenvalue = sorted_eigenvalues[0]
    condition_indices = [np.sqrt(max_eigenvalue / ev) for ev in sorted_eigenvalues]
    
    # Step 5: Find the maximum Condition Index
    max_condition_index = max(condition_indices)
    
    # Step 5: Return condition indices as a DataFrame
    condition_index_df = pd.DataFrame({
        'Eigenvalue': sorted_eigenvalues,
        'Condition Index': condition_indices
    })

    return condition_index_df, max_condition_index


3

In [4]:
def calculate_vif1(df):
    # Initialize a DataFrame to store the VIF for each variable
    vif_data = pd.DataFrame()
    vif_data["Variable"] = df.columns
    vif_values = []

    for i in range(df.shape[1]):
        # Take the i-th column as the dependent variable, and the remaining columns as independent variables
        y = df.iloc[:, i]
        X = df.drop(df.columns[i], axis=1)

        # Calculate the R^2 value: beta = (X^T X)^(-1) X^T y
        X = np.column_stack([np.ones(X.shape[0]), X])  # Add an intercept term
        beta = np.linalg.lstsq(X, y, rcond=None)[0]  # Solve for the least squares solution
        y_pred = X @ beta  # Predicted values
        r_squared = 1 - np.sum((y - y_pred) ** 2) / np.sum((y - y.mean()) ** 2)

        # Calculate the VIF value
        vif = 1 / (1 - r_squared)
        vif_values.append(vif)

    vif_data["VIF"] = vif_values
    return vif_data



4

In [5]:
def calculate_variance_decomposition(df):
    # Step 1: Calculate the covariance matrix
    cov_matrix = df.cov()

    # Step 2: Calculate eigenvalues and eigenvectors of the covariance matrix
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

    # Step 3: Compute the variance decomposition proportions for each variable
    variance_decomposition = []
    for i in range(len(df.columns)):
        # Compute the variance proportion of the i-th variable for each eigenvalue
        proportions = (eigenvectors[i, :] ** 2) * eigenvalues / cov_matrix.iloc[i, i]
        variance_decomposition.append(proportions / proportions.sum())

    # Convert results to a DataFrame
    variance_decomposition_df = pd.DataFrame(
        variance_decomposition, 
        columns=[f'Eigenvalue_{j+1}' for j in range(len(eigenvalues))],
        index=df.columns
    )
    return variance_decomposition_df

5

In [6]:
def calculate_determinant_of_correlation_matrix(df):
    # Step 1: Compute the correlation matrix
    corr_matrix = df.corr()

    # Step 2: Compute eigenvalues of the correlation matrix
    eigenvalues = np.linalg.eigvals(corr_matrix)

    # Step 3: Calculate the determinant as the product of the eigenvalues
    determinant = np.prod(eigenvalues)

    return determinant

第一个数据集共线性不强，不必分析

In [12]:
file_path = 'METHOD_BUILT_ALL_RHO_MATRIX.xlsx'  # 替换为你的文件路径
df1 = pd.read_excel(file_path)
df1 = df1.apply(np.log)
df1.fillna(df1.mean(), inplace=True)  # 用均值填充
df1.fillna(df1.median(), inplace=True)  # 用中位数填充

print(df1.isna().sum())  # 检查每列 NaN 值的数量
print(np.isinf(df1).sum())  # 检查 Inf 值

print(PC(df1))
# Compute condition index
cond_index, max_condition_index = calculate_condition_index(df1)
print(cond_index)
print(max_condition_index)
r = calculate_vif1(df1)
print(r)
vd_df = calculate_variance_decomposition(df1)
print(vd_df)
D = calculate_determinant_of_correlation_matrix(df1)
print("Determinant of the Correlation Matrix (D):", D)

y     0
x1    0
x2    0
x3    0
x4    0
dtype: int64
y     0
x1    0
x2    0
x3    0
x4    0
dtype: int64
           y        x1        x2        x3        x4
y   1.000000  0.562771  0.296347  0.403551  0.376560
x1  0.562771  1.000000  0.213782  0.738732  0.159443
x2  0.296347  0.213782  1.000000  0.538900  0.616268
x3  0.403551  0.738732  0.538900  1.000000  0.317926
x4  0.376560  0.159443  0.616268  0.317926  1.000000
   Eigenvalue  Condition Index
0    2.704458         1.000000
1    1.132885         1.545067
2    0.681507         1.992072
3    0.332484         2.852036
4    0.148667         4.265142
4.265141555638201
  Variable       VIF
0        y  1.721033
1       x1  3.258947
2       x2  2.288762
3       x3  3.429646
4       x4  1.755539
    Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4  Eigenvalue_5
y       0.224181      0.093809      0.062336  3.014969e-01      0.318177
x1      0.383073      0.594317      0.022584  8.351008e-06      0.000017
x2      0.795414      0.201



第二个有了一定共线性

In [13]:
file_path = 'METHOD_DIFF_FUN_1.xlsx'  # 替换为你的文件路径
df1 = pd.read_excel(file_path)
df1 = df1[['x1','x2','x3',"x4"]] 
print(PC(df1))
# Compute condition index
cond_index, max_condition_index = calculate_condition_index(df1)
print(cond_index)
print(max_condition_index)
r = calculate_vif1(df1)
print(r)
vd_df = calculate_variance_decomposition(df1)
print(vd_df)
D = calculate_determinant_of_correlation_matrix(df1)
print("Determinant of the Correlation Matrix (D):", D)

          x1        x2        x3        x4
x1  1.000000 -0.005156 -0.017734 -0.024587
x2 -0.005156  1.000000  0.008771 -0.019293
x3 -0.017734  0.008771  1.000000 -0.001354
x4 -0.024587 -0.019293 -0.001354  1.000000
   Eigenvalue  Condition Index
0    1.030563         1.000000
1    1.019556         1.005384
2    0.985443         1.022637
3    0.964438         1.033714
1.0337135183359456
  Variable       VIF
0       x1  1.000951
1       x2  1.000479
2       x3  1.000393
3       x4  1.000985
    Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4
x1      0.000360  7.571344e-03      0.991898  1.704758e-04
x2      0.000085  1.396154e-03      0.000456  9.980630e-01
x3      0.999998  1.601077e-09      0.000002  1.550020e-07
x4      0.000002  9.957527e-01      0.003871  3.745813e-04
Determinant of the Correlation Matrix (D): 0.9985997435528587


第三个

In [14]:
for i in range(10):
    file_path = 'METHOD_DIFF_FUN_'+str(i+1)+'.xlsx'  # 替换为你的文件路径
    df1 = pd.read_excel(file_path)
    df1 = df1[['x1','x2','x3',"x4"]] 
    print("==============================================================================")
    print("file_path")
    print(PC(df1))
    # Compute condition index
    cond_index, max_condition_index = calculate_condition_index(df1)
    print(cond_index)
    print(max_condition_index)
    r = calculate_vif1(df1)
    print(r)
    vd_df = calculate_variance_decomposition(df1)
    print(vd_df)
    D = calculate_determinant_of_correlation_matrix(df1)
    print("Determinant of the Correlation Matrix (D):", D)
    print("==============================================================================")

file_path
          x1        x2        x3        x4
x1  1.000000 -0.005156 -0.017734 -0.024587
x2 -0.005156  1.000000  0.008771 -0.019293
x3 -0.017734  0.008771  1.000000 -0.001354
x4 -0.024587 -0.019293 -0.001354  1.000000
   Eigenvalue  Condition Index
0    1.030563         1.000000
1    1.019556         1.005384
2    0.985443         1.022637
3    0.964438         1.033714
1.0337135183359456
  Variable       VIF
0       x1  1.000951
1       x2  1.000479
2       x3  1.000393
3       x4  1.000985
    Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4
x1      0.000360  7.571344e-03      0.991898  1.704758e-04
x2      0.000085  1.396154e-03      0.000456  9.980630e-01
x3      0.999998  1.601077e-09      0.000002  1.550020e-07
x4      0.000002  9.957527e-01      0.003871  3.745813e-04
Determinant of the Correlation Matrix (D): 0.9985997435528587
file_path
          x1        x2        x3        x4
x1  1.000000  0.694113  0.488116  0.323561
x2  0.694113  1.000000  0.699115  0.485545


In [18]:
for i in range(8):
    file_path = 'METHOD_MULTICOLLINEAR_'+str(i+1)+'.xlsx'  # 替换为你的文件路径
    df1 = pd.read_excel(file_path)
    #df1 = df1[['x1','x2','x3',"x4"]] 
    print("==============================================================================")
    print("file_path")
    print(PC(df1))
    # Compute condition index
    cond_index, max_condition_index = calculate_condition_index(df1)
    print(cond_index)
    print(max_condition_index)
    r = calculate_vif1(df1)
    print(r)
    vd_df = calculate_variance_decomposition(df1)
    print(vd_df)
    D = calculate_determinant_of_correlation_matrix(df1)
    print("Determinant of the Correlation Matrix (D):", D)
    print("==============================================================================")

file_path
           y        x1        x2        x3        x4
y   1.000000  0.767996  0.594476  0.644954  0.407327
x1  0.767996  1.000000  0.261934  0.970686 -0.036633
x2  0.594476  0.261934  1.000000  0.041787 -0.003915
x3  0.644954  0.970686  0.041787  1.000000 -0.037126
x4  0.407327 -0.036633 -0.003915 -0.037126  1.000000
   Eigenvalue  Condition Index
0    2.765788         1.000000
1    1.212812         1.510124
2    0.979183         1.680651
3    0.039265         8.392772
4    0.002952        30.609279
30.60927881731557
  Variable         VIF
0        y   22.723543
1       x1  193.685117
2       x2    7.232054
3       x3  137.522032
4       x4    5.268630
    Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4  Eigenvalue_5
y       0.726699      0.272461      0.000284  5.305182e-04      0.000025
x1      0.978438      0.020355      0.000961  5.173135e-05      0.000194
x2      0.111280      0.390011      0.443291  4.850338e-02      0.006914
x3      0.886895      0.094287      0.

Determinant of the Correlation Matrix (D): 1.7335502096152406e-07
file_path
           y        x1        x2        x3        x4
y   1.000000  0.789467  0.521054  0.695491  0.311694
x1  0.789467  1.000000  0.144583  0.969824 -0.071267
x2  0.521054  0.144583  1.000000 -0.068293 -0.025115
x3  0.695491  0.969824 -0.068293  1.000000  0.008433
x4  0.311694 -0.071267 -0.025115  0.008433  1.000000
   Eigenvalue  Condition Index
0    2.718454         1.000000
1    1.195309         1.508068
2    1.036386         1.619572
3    0.046744         7.626056
4    0.003107        29.580762
29.580761737351335
  Variable         VIF
0        y   18.860196
1       x1  181.332012
2       x2    6.270881
3       x3  133.277213
4       x4    5.704340
    Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4  Eigenvalue_5
y       0.735895      0.263001      0.000334      0.000740      0.000030
x1      0.984197      0.013954      0.001635      0.000044      0.000169
x2      0.041960      0.503419      0.361216

In [26]:
file_path = 'METHOD_4_CHANGE_COLL.xlsx'  # 替换为你的文件路径
df1 = pd.read_excel(file_path)
print(PC(df1))
# Compute condition index
cond_index, max_condition_index = calculate_condition_index(df1)
print(cond_index)
print(max_condition_index)
r = calculate_vif1(df1)
print(r)
vd_df = calculate_variance_decomposition(df1)
print(vd_df)
D = calculate_determinant_of_correlation_matrix(df1)
print("Determinant of the Correlation Matrix (D):", D)

                    y        x1        x2        x3        x4  x1_quantile
y            1.000000  0.698790  0.687545  0.473542  0.262893     0.055235
x1           0.698790  1.000000  0.174031  0.051760  0.008612     0.089940
x2           0.687545  0.174031  1.000000  0.485607  0.247285     0.000137
x3           0.473542  0.051760  0.485607  1.000000  0.662670     0.096422
x4           0.262893  0.008612  0.247285  0.662670  1.000000     0.099667
x1_quantile  0.055235  0.089940  0.000137  0.096422  0.099667     1.000000
   Eigenvalue  Condition Index
0    2.593054         1.000000
1    1.342760         1.389654
2    1.017041         1.596749
3    0.677667         1.956130
4    0.282147         3.031574
5    0.087331         5.449059
5.449059055019367
      Variable       VIF
0            y  6.765839
1           x1  3.529183
2           x2  2.926516
3           x3  2.526026
4           x4  1.810283
5  x1_quantile  1.026367
             Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue

In [27]:
file_path = 'METHOD_5_SCALE_DIFF.xlsx'  # 替换为你的文件路径
df1 = pd.read_excel(file_path)
print(PC(df1))
# Compute condition index
cond_index, max_condition_index = calculate_condition_index(df1)
print(cond_index)
print(max_condition_index)
r = calculate_vif1(df1)
print(r)
vd_df = calculate_variance_decomposition(df1)
print(vd_df)
D = calculate_determinant_of_correlation_matrix(df1)
print("Determinant of the Correlation Matrix (D):", D)

          x1        x2        x3        x4         y
x1  1.000000  0.886747  0.891387  0.900302  0.976288
x2  0.886747  1.000000  0.885052  0.900011  0.956386
x3  0.891387  0.885052  1.000000  0.926413  0.946169
x4  0.900302  0.900011  0.926413  1.000000  0.944224
y   0.976288  0.956386  0.946169  0.944224  1.000000
   Eigenvalue  Condition Index
0    4.686248         1.000000
1    0.125844         6.102340
2    0.114779         6.389707
3    0.072884         8.018580
4    0.000245       138.268131
138.26813095442876
  Variable          VIF
0       x1   691.826025
1       x2   302.023868
2       x3   122.657413
3       x4    16.374948
4        y  2977.322900
    Eigenvalue_1  Eigenvalue_2  Eigenvalue_3  Eigenvalue_4  Eigenvalue_5
x1      0.951914      0.000433      0.041666      0.005776      0.000211
x2      0.915587      0.000197      0.072935      0.011265      0.000016
x3      0.896425      0.000174      0.001589      0.092284      0.009529
x4      0.893270      0.000023      0.003

Determinant 不是好的选择